### This notebook performs data cleaning and standardization to create
### the Silver layer.

In the Silver layer, raw COVID-19 and vaccination data is cleaned, validated, and standardized to improve data quality.  
This includes schema validation, handling missing values, and applying consistent data formats to prepare reliable datasets for feature engineering and machine learning.


Load Bronze data again

In [0]:
covid_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/default/raw_data/COVID-19_India_Data.csv")
)

vaccine_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/default/raw_data/Vaccination.csv")
)


Inspect schema

In [0]:
covid_df.printSchema()


root
 |-- date: date (nullable = true)
 |-- new_cases: integer (nullable = true)
 |-- cum_cases: integer (nullable = true)
 |-- new_death: integer (nullable = true)
 |-- cum_death: integer (nullable = true)
 |-- new_recovered: integer (nullable = true)
 |-- cum_recovered: integer (nullable = true)
 |-- cum_active_cases: integer (nullable = true)



In [0]:
vaccine_df.printSchema()


root
 |-- date: date (nullable = true)
 |-- total_vaccinations: integer (nullable = true)
 |-- people_vaccinated: integer (nullable = true)
 |-- people_fully_vaccinated: integer (nullable = true)
 |-- daily_vaccinations_raw: integer (nullable = true)
 |-- daily_vaccinations: integer (nullable = true)
 |-- total_vaccinations_per_hundred: double (nullable = true)
 |-- people_vaccinated_per_hundred: double (nullable = true)



Column names were reviewed and standardized in the Silver layer to ensure consistent naming conventions across all datasets.



In [0]:
from pyspark.sql.functions import col

def standardize_columns(df):
    for column in df.columns:
        df = df.withColumnRenamed(column, column.lower())
    return df

covid_clean_df = standardize_columns(covid_df)
vaccine_clean_df = standardize_columns(vaccine_df)


This step is included to ensure pipeline robustness and reproducibility.

In [0]:
covid_clean_df.columns


['date',
 'new_cases',
 'cum_cases',
 'new_death',
 'cum_death',
 'new_recovered',
 'cum_recovered',
 'cum_active_cases']

In [0]:
vaccine_clean_df.columns


['date',
 'total_vaccinations',
 'people_vaccinated',
 'people_fully_vaccinated',
 'daily_vaccinations_raw',
 'daily_vaccinations',
 'total_vaccinations_per_hundred',
 'people_vaccinated_per_hundred']

### Handling Missing Values

In [0]:
from pyspark.sql.functions import col, sum

covid_clean_df.select(
    [sum(col(c).isNull().cast("int")).alias(c) for c in covid_clean_df.columns]
).show()


+----+---------+---------+---------+---------+-------------+-------------+----------------+
|date|new_cases|cum_cases|new_death|cum_death|new_recovered|cum_recovered|cum_active_cases|
+----+---------+---------+---------+---------+-------------+-------------+----------------+
|   0|        0|        0|        0|        0|            0|            0|               0|
+----+---------+---------+---------+---------+-------------+-------------+----------------+



In [0]:
vaccine_clean_df.select(
    [sum(col(c).isNull().cast("int")).alias(c) for c in vaccine_clean_df.columns]
).show()


+----+------------------+-----------------+-----------------------+----------------------+------------------+------------------------------+-----------------------------+
|date|total_vaccinations|people_vaccinated|people_fully_vaccinated|daily_vaccinations_raw|daily_vaccinations|total_vaccinations_per_hundred|people_vaccinated_per_hundred|
+----+------------------+-----------------+-----------------------+----------------------+------------------+------------------------------+-----------------------------+
|   0|                11|               11|                     40|                    22|                 1|                            11|                           11|
+----+------------------+-----------------+-----------------------+----------------------+------------------+------------------------------+-----------------------------+



Missing values are handled using forward fill for cumulative vaccination metrics and zero fill for daily counts.


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import last

# Window specification for forward fill
window_spec = Window.orderBy("date").rowsBetween(Window.unboundedPreceding, 0)

vaccine_filled_df = (
    vaccine_clean_df
    .withColumn("total_vaccinations", last("total_vaccinations", True).over(window_spec))
    .withColumn("people_vaccinated", last("people_vaccinated", True).over(window_spec))
    .withColumn("people_fully_vaccinated", last("people_fully_vaccinated", True).over(window_spec))
    .withColumn("total_vaccinations_per_hundred", last("total_vaccinations_per_hundred", True).over(window_spec))
    .withColumn("people_vaccinated_per_hundred", last("people_vaccinated_per_hundred", True).over(window_spec))
    .fillna({
        "daily_vaccinations_raw": 0,
        "daily_vaccinations": 0
    })
)


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
vaccine_filled_df.select(
    [sum(col(c).isNull().cast("int")).alias(c) for c in vaccine_filled_df.columns]
).show()


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+----+------------------+-----------------+-----------------------+----------------------+------------------+------------------------------+-----------------------------+
|date|total_vaccinations|people_vaccinated|people_fully_vaccinated|daily_vaccinations_raw|daily_vaccinations|total_vaccinations_per_hundred|people_vaccinated_per_hundred|
+----+------------------+-----------------+-----------------------+----------------------+------------------+------------------------------+-----------------------------+
|   0|                 0|                0|                     29|                     0|                 0|                             0|                            0|
+----+------------------+-----------------+-----------------------+----------------------+------------------+------------------------------+-----------------------------+



### Save Cleaned Data as Silver Tables

In [0]:
covid_clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.covid_silver")


In [0]:
vaccine_filled_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.vaccination_silver")


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
spark.sql("SHOW TABLES IN workspace.default").show()


+--------+------------------+-----------+
|database|         tableName|isTemporary|
+--------+------------------+-----------+
| default|      covid_silver|      false|
| default|vaccination_silver|      false|
+--------+------------------+-----------+

